# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [2]:
# Importation des packages nécessaires

import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [3]:
# Chargement des données
df_mapping_initial = pd.read_csv("../data_finale/mapping_fbref_tm.csv", encoding='latin1')
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")
df_tm_initial = pd.read_csv("../data/transfermarkt_datasets/players.csv")

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [4]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [5]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm)

[1] Nom exact (mapping)     : 17121 | restants : 833
[2] Fuzzy nom (mapping)     :    55 | restants : 778


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [6]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')